# Train a cigarette-butt detector

Trains one YOLO detector from one **recipe** on the
[Cigarette Litter Street Photo Dataset](https://www.kaggle.com/datasets/lorinc/butt-field-2026-10-01)
(125 phone photos of city pavement, 4000×1800 px, cigarette butts labelled by hand).

A recipe is a JSON file in `pipeline/recipes/` holding every setting of a run: how photos are cut into 640 px
training tiles, which tiles are kept, the model and its training arguments. The notebook never changes a recipe;
it writes the settings it actually used to `run_config.json` next to the trained weights, and the evaluation
notebook reads that file.

| Recipe | Model | What it adds |
|---|---|---|
| `03-cv-baseline.json` | YOLO11s | `maybe` boxes (objects the annotator was unsure of) painted grey, 10% of training tiles butt-free |
| `04-cv-hard-negatives.json` | YOLO11s | plus tiles centred on lookalikes the annotator rejected |
| `05-cv-more-empty-tiles.json` | YOLO11s | 03 with random butt-free tiles in 04's share instead |
| `06-cv-yolo11n.json` | YOLO11n | 04's recipe on the small model, about 3 times faster |

The model trains on the 87 training photos; the 19 validation and 19 test photos stay unseen. This is the same
training set as fold 0 of the cross-validation, whose models run in the
[web demo](https://lorinc.github.io/litter-detection-pipeline/web_demo/).

**On Kaggle:** add the dataset `lorinc/butt-field-2026-10-01` as input, choose the GPU T4 accelerator and turn
internet on (the notebook clones the code). A full run of 100 epochs takes about 70 minutes for YOLO11s on one T4.
**Elsewhere:** run it from the `pipeline/` folder of a clone and set `DATA` to a copy of the dataset.

## Settings

In [ ]:
RECIPE = 'recipes/04-cv-hard-negatives.json'  # relative to pipeline/
REPO = 'https://github.com/lorinc/litter-detection-pipeline.git'
COMMIT = 'main'       # branch, tag or commit of the code to clone
DATA = None           # dataset folder (images/, labels/, split.json); None finds it under /kaggle/input
WORK = None           # output folder; None: /kaggle/working on Kaggle, ./work elsewhere
SMOKE_TEST = False    # True: 1 epoch on 30% of the training tiles, to check the setup in a few minutes

## Code, dataset and dependencies

In [ ]:
import glob, json, os, pathlib, shutil, subprocess, sys

def sh(cmd, cwd=None, env=None):
    """Run a shell command, streaming its output into the notebook; stop on failure."""
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, env=env, text=True,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in p.stdout:
        print(line, end='')
    if p.wait():
        raise RuntimeError(f'exit {p.returncode}: {cmd}')

ON_KAGGLE = os.path.isdir('/kaggle/input')
if os.path.isfile('train.py') and os.path.isfile('export.py'):
    CODE = pathlib.Path.cwd()  # running inside a clone's pipeline/ folder
else:
    clone = pathlib.Path('/tmp/litter-detection-pipeline')
    if not clone.exists():
        sh(f'git clone -q {REPO} {clone}')
    sh(f'git -C {clone} checkout -q {COMMIT}')
    CODE = clone / 'pipeline'
if DATA is None:
    found = glob.glob('/kaggle/input/**/split.json', recursive=True)
    assert found, 'dataset not found: add lorinc/butt-field-2026-10-01 as input, or set DATA'
    DATA = os.path.dirname(found[0])
WORK = pathlib.Path(WORK or ('/kaggle/working' if ON_KAGGLE else 'work')).resolve()

recipe = json.load(open(CODE / RECIPE))
version = recipe['train']['ultralytics']
try:
    import ultralytics
    installed = ultralytics.__version__
except ImportError:
    installed = None
if installed != version:
    sh(f'{sys.executable} -m pip install -q ultralytics=={version}')
print(f'code   {CODE} ({subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=CODE, capture_output=True, text=True).stdout.strip()})')
print(f'data   {DATA}')
print(f'output {WORK}')

## Run configuration

The recipe plus the evaluation entry for this run (where its weights will be, the tiling the harness uses),
saved as `run_config.json`.

In [ ]:
RUN = pathlib.Path(RECIPE).stem
out = WORK / RUN
out.mkdir(parents=True, exist_ok=True)
run_cfg = json.loads(json.dumps(recipe))
X, T = run_cfg['export'], run_cfg['train']
run_cfg['eval']['detectors'][RUN] = {**run_cfg['run']['eval'], 'weights': str(out / 'train/weights/last.pt'),
                                     'tile': X['tile'], 'stride': X['stride'], 'imgsz': X['tile']}
if SMOKE_TEST:
    T['args'] |= {'epochs': 1, 'fraction': 0.3}
CONFIG = out / 'run_config.json'
json.dump(run_cfg, open(CONFIG, 'w'), indent=2)

print(f'run {RUN}: {T["model"]}, Ultralytics {T["ultralytics"]}, training arguments {T["args"]}')
print(f'tiles {X["tile"]} px every {X["stride"]} px; a box counts when {X["min_visible"]:.0%} of it is inside a tile')
print(f'maybe boxes: {X["maybe"]}; butt-free share of training tiles: {X["background_share"]}; '
      f'hard negatives: {X["hard_negatives"]}')
print(f'saved {CONFIG}')

## Export training tiles

Cuts every photo into overlapping 640 px tiles with YOLO labels, following the recipe. The export goes to a
temporary folder outside the saved output; only its index, `export.json`, is kept with the run.

In [ ]:
EXPORT = pathlib.Path('/tmp') / f'export_{RUN}'
sh(f'{sys.executable} export.py --config {CONFIG} --data {DATA} --out {EXPORT}', cwd=CODE)
shutil.copy(EXPORT / 'export.json', out / 'export.json')

## Train

Ultralytics training on one GPU (on a CPU if there is none, very slowly). There is no validation during training
and no early stopping: the final epoch's `last.pt` is the model, so no scored photo picks the checkpoint.

In [ ]:
sh(f'{sys.executable} train.py --config {CONFIG} --data {EXPORT} --project {out}', cwd=CODE,
   env={**os.environ, 'CUDA_VISIBLE_DEVICES': os.environ.get('CUDA_VISIBLE_DEVICES', '0')})
shutil.rmtree(EXPORT, ignore_errors=True)

## Result

In [ ]:
import csv
rows = list(csv.DictReader(open(out / 'train/results.csv')))
last = {k.strip(): v for k, v in rows[-1].items()}
print(f'{len(rows)} epochs; final losses: box {float(last["train/box_loss"]):.3f}, '
      f'class {float(last["train/cls_loss"]):.3f}, dfl {float(last["train/dfl_loss"]):.3f}')
print(f'weights        {out / "train/weights/last.pt"}')
print(f'configuration  {CONFIG}')

Next: `evaluate.ipynb` scores `last.pt` on whole validation photos and draws the recall against false-positives
curve. On Kaggle, save this notebook's version and add its output as an input of the evaluation notebook.